### Experiment Notebook
Day-ahead ERCOT South Central load forecasting — SHAP-guided feature engineering, reproducing the source paper's experiment sequence.


#### Environment setup
Load credentials from environment variables

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['MLFLOW_TRACKING_URI'] = os.getenv("MLFLOW_TRACKING_URI")
os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv("MLFLOW_TRACKING_PASSWORD")

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))  # if notebook is in a subfolder
sys.path.insert(0, str(Path.cwd() / "scripts"))          # if notebook is at the project root

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# shared metric + feature-engineering helpers (scripts/metrics_utils.py, scripts/engineered_features.py)
from metrics_utils import compute_all_metrics


In [2]:
import mlflow

mlflow.set_experiment('electricity-distribution-forecast')


c:\Users\hp\Documents\ML_AI_Projects\electricity-distribution-forecast\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/7f3415414c0a400c827e6977c0ec79cc', creation_time=1785420407673, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785420407673, lifecycle_stage='active', name='electricity-distribution-forecast', tags={}, trace_location=None, workspace='default'>

#### 1. Load raw joined dataset
Load pack (GridStatus load pull) + weather (Open-Meteo population-weighted) - already joined by the existing pull/concat scripts.

In [3]:
#load the joined raw dataframe (from DVC-tracked storage / local cache)
df_raw = pd.read_csv("../data/raw/ercot_south_central_raw.csv")


#### 2. Validate raw data with Great Expectations


In [4]:
from validate_raw_data import validate_dataframe
# df is the DataFrame you already loaded
passed = validate_dataframe(df_raw)
if not passed:
    raise RuntimeError("Data failed validation — fix issues before proceeding.")

2026-07-30 17:07:16  INFO      ================================================================
2026-07-30 17:07:16  INFO      GX Validation Gate  |  in-memory DataFrame  (92687 rows × 5 cols)
2026-07-30 17:07:16  INFO      ================================================================
2026-07-30 17:07:16  INFO      Source column names: ['timestamp', 'actual_load_mw', 'temp_c', 'humidity_pct', 'precip_mm']
2026-07-30 17:07:16  INFO      Columns renamed to canonical names: ['timestamp', 'actual_load_mw', 'temp_c', 'humidity_pct', 'precip_mm']
2026-07-30 17:07:16  INFO      Created temporary directory 'C:\Users\hp\AppData\Local\Temp\tmpe__p2m5b' for ephemeral docs site
2026-07-30 17:07:16  INFO      Loading 'datasources' ->
[]
2026-07-30 17:07:16  INFO      ExpectationSuite 'raw_data_validation_suite' registered (13 expectations).
2026-07-30 17:07:16  INFO      Running checkpoint …
Calculating Metrics: 100%|██████████| 55/55 [00:00<00:00, 128.95it/s]
2026-07-30 17:07:17  INFO      ====

#### 3. Baseline feature set
Calendar features (`hour`, `dayofweek`, `month`) + weather (`tavg`, `tmin`, `tmax`, `prcp`) + lagged load (`load_lag_24`, `load_lag_168`).

In [6]:
from feature_engineering import add_baseline_feature_set

df_feat = add_baseline_feature_set(df_raw)


2026-07-30 17:09:19  INFO      add_baseline_feature_set: 92687 rows × 19 columns.


In [7]:
df_feat.head()

,timestamp,actual_load_mw,temp_c,humidity_pct,precip_mm,hour,day_of_week,month,is_weekend,is_holiday,load_lag_24,load_lag_168,load_roll_mean_24,load_roll_max_24,load_roll_std_24,tavg,prcp,tmax,tmin
0,2016-01-01 00:00:00+00:00,6848.58,10.898413,60.445621,0.0,0,4,1,0,0,NaN,NaN,NaN,NaN,NaN,10.898413,0.0,NaN,NaN
1,2016-01-01 01:00:00+00:00,6599.73,10.706991,60.524286,0.0,1,4,1,0,0,NaN,NaN,NaN,NaN,NaN,10.706991,0.0,NaN,NaN
2,2016-01-01 02:00:00+00:00,6357.69,10.583955,61.417780,0.0,2,4,1,0,0,NaN,NaN,NaN,NaN,NaN,10.583955,0.0,NaN,NaN
3,2016-01-01 03:00:00+00:00,6120.83,10.394830,61.863401,0.0,3,4,1,0,0,NaN,NaN,NaN,NaN,NaN,10.394830,0.0,NaN,NaN
4,2016-01-01 04:00:00+00:00,5882.63,10.189748,62.309023,0.0,4,4,1,0,0,NaN,NaN,NaN,NaN,NaN,10.189748,0.0,NaN,NaN


In [8]:
baseline_cols = [
    "hour", "day_of_week", "month",    # calendar
    "tavg", "tmin", "tmax", "prcp",    # weather (spec names)
    "load_lag_24", "load_lag_168",     # lagged load
]
df_feat["timestamp"] = pd.to_datetime(df_feat["timestamp"])
df_feat = df_feat.sort_values("timestamp").reset_index(drop=True)

df_baseline = df_feat[["timestamp", "actual_load_mw"] + baseline_cols].copy()
df_baseline.head()


,timestamp,actual_load_mw,hour,day_of_week,month,tavg,tmin,tmax,prcp,load_lag_24,load_lag_168
0,2016-01-01 00:00:00+00:00,6848.58,0,4,1,10.898413,NaN,NaN,0.0,NaN,NaN
1,2016-01-01 01:00:00+00:00,6599.73,1,4,1,10.706991,NaN,NaN,0.0,NaN,NaN
2,2016-01-01 02:00:00+00:00,6357.69,2,4,1,10.583955,NaN,NaN,0.0,NaN,NaN
3,2016-01-01 03:00:00+00:00,6120.83,3,4,1,10.394830,NaN,NaN,0.0,NaN,NaN
4,2016-01-01 04:00:00+00:00,5882.63,4,4,1,10.189748,NaN,NaN,0.0,NaN,NaN


#### Experiment 1 - Linear Regression

### 1a. Baseline Linear Regression

In [ ]:
target_col = "actual_load_mw"
feature_cols_baseline = baseline_cols

# Paper trains on 2016-2023, tests on 2024 — reused by the XGBoost/LightGBM
# baseline cells below since they share the same baseline feature set.
train_mask = df_baseline["timestamp"].dt.year <= 2023
test_mask = df_baseline["timestamp"].dt.year == 2024

X_train = df_baseline.loc[train_mask, feature_cols_baseline]
X_test = df_baseline.loc[test_mask, feature_cols_baseline]
y_train = df_baseline.loc[train_mask, target_col]
y_test = df_baseline.loc[test_mask, target_col]

# LR is scale-sensitive; the tree models below reuse the unscaled X_train/X_test directly.
lr_scaler = StandardScaler()
X_train_lr = lr_scaler.fit_transform(X_train)
X_test_lr = lr_scaler.transform(X_test)

with mlflow.start_run(run_name='lr_baseline') as run:
    lr_baseline_model = LinearRegression()
    lr_baseline_model.fit(X_train_lr, y_train)
    lr_baseline_preds = lr_baseline_model.predict(X_test_lr)

    lr_baseline_metrics = compute_all_metrics(y_test.values, lr_baseline_preds)
    print(f"LR baseline -> MAPE {lr_baseline_metrics['mape']:.2f}% | RMSE {lr_baseline_metrics['rmse']:.2f} | "
          f"MAE {lr_baseline_metrics['mae']:.2f} | Peak-MAPE {lr_baseline_metrics['peak_mape']:.2f}%")

    mlflow.log_params({"model": "LinearRegression", "feature_set": "baseline", "n_features": len(feature_cols_baseline)})
    mlflow.log_metrics(lr_baseline_metrics)
    mlflow.sklearn.log_model(lr_baseline_model, "model")

    lr_baseline_run_id = run.info.run_id


### 1b. SHAP diagnosis on baseline LR
Expect `load_lag_24`, `load_roll_mean_24`, `load_lag_168` to dominate; static calendar features to contribute little.

In [ ]:
explainer_lr = shap.LinearExplainer(lr_baseline_model, X_train_lr, feature_names=feature_cols_baseline)
shap_values_lr = explainer_lr(X_test_lr)

shap.summary_plot(shap_values_lr, X_test_lr, feature_names=feature_cols_baseline, show=False)
plt.tight_layout()
plt.savefig("shap_lr_baseline.png", dpi=150)
plt.close()

with mlflow.start_run(run_id=lr_baseline_run_id):
    mlflow.log_artifact("shap_lr_baseline.png")


### 1c. Engineer SHAP-guided features for LR
Add: `CDD_lag_24`, `HDD_lag_24`, `temp_spike_vs_mean`, `is_extreme_cold_event`, `is_extreme_heat_event`, `hour_sin`, `dayofweek_cos`, `lag_24_x_hour`, `CDD_x_hour`

In [ ]:
from engineered_features import build_lr_engineered_features

df_lr_v2 = build_lr_engineered_features(df_feat)
lr_engineered_cols = [c for c in df_lr_v2.columns if c not in ("timestamp", target_col)]
df_lr_v2.head()


### 1d. Retrain improved LR
Target: MAPE approx 4.74% (~11% relative improvement)

In [ ]:
train_mask_v2 = df_lr_v2["timestamp"].dt.year <= 2023
test_mask_v2 = df_lr_v2["timestamp"].dt.year == 2024

X_train_v2 = df_lr_v2.loc[train_mask_v2, lr_engineered_cols]
X_test_v2 = df_lr_v2.loc[test_mask_v2, lr_engineered_cols]
y_train_v2 = df_lr_v2.loc[train_mask_v2, target_col]
y_test_v2 = df_lr_v2.loc[test_mask_v2, target_col]

lr_v2_scaler = StandardScaler()
X_train_lr_v2 = lr_v2_scaler.fit_transform(X_train_v2)
X_test_lr_v2 = lr_v2_scaler.transform(X_test_v2)

with mlflow.start_run(run_name='lr_shap_engineered') as run:
    lr_v2_model = LinearRegression()
    lr_v2_model.fit(X_train_lr_v2, y_train_v2)
    lr_v2_preds = lr_v2_model.predict(X_test_lr_v2)

    lr_v2_metrics = compute_all_metrics(y_test_v2.values, lr_v2_preds)
    print(f"LR + SHAP features -> MAPE {lr_v2_metrics['mape']:.2f}% (baseline was {lr_baseline_metrics['mape']:.2f}%)")

    mlflow.log_params({"model": "LinearRegression", "feature_set": "shap_engineered", "n_features": len(lr_engineered_cols)})
    mlflow.log_metrics(lr_v2_metrics)
    mlflow.sklearn.log_model(lr_v2_model, "model")

    lr_v2_run_id = run.info.run_id


---
## Experiment 2 - XGBoost (paper's best performer)

### 2a. Baseline XGBoost
Target: MAPE approx 3.15% (your earlier reproduction landed 3.35% val - close enough to validate the pipeline; revisit the val to test gap noted earlier if it persists here)

In [ ]:
from xgboost import XGBRegressor

with mlflow.start_run(run_name='xgb_baseline') as run:
    xgb_baseline_model = XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1,
    )
    xgb_baseline_model.fit(X_train, y_train)
    xgb_baseline_preds = xgb_baseline_model.predict(X_test)

    xgb_baseline_metrics = compute_all_metrics(y_test.values, xgb_baseline_preds)
    print(f"XGB baseline -> MAPE {xgb_baseline_metrics['mape']:.2f}% | Peak-MAPE {xgb_baseline_metrics['peak_mape']:.2f}%")

    mlflow.log_params({"model": "XGBRegressor", "feature_set": "baseline", "n_features": len(feature_cols_baseline)})
    mlflow.log_metrics(xgb_baseline_metrics)
    mlflow.xgboost.log_model(xgb_baseline_model, "model")

    xgb_baseline_run_id = run.info.run_id


### 2b. SHAP diagnosis on baseline XGBoost
Expect underprediction during peak events (early-morning winter, late-afternoon summer); `month`/`dayofweek` dominating importance despite being static.

In [ ]:
explainer_xgb = shap.TreeExplainer(xgb_baseline_model)
shap_values_xgb = explainer_xgb(X_test)

shap.summary_plot(shap_values_xgb, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_xgb_baseline.png", dpi=150)
plt.close()

with mlflow.start_run(run_id=xgb_baseline_run_id):
    mlflow.log_artifact("shap_xgb_baseline.png")


### 2c. Engineer SHAP-guided features for XGBoost
Add:
- load_spike_vs_mean = (load - load_roll_mean_24) / (load_roll_mean_24 + 1)
- temp_spike_vs_mean = (tmax - tavg) / (tavg + 1)
- tmax_roll_max_72
- CDD_x_hour = CDD x hour
- lag_24_x_hour = load_lag_24 x hour
- is_extreme_heat_event = 1[tmax > 95th percentile]
- is_monday

In [ ]:
from engineered_features import build_xgb_engineered_features

df_xgb_v2 = build_xgb_engineered_features(df_feat)
xgb_engineered_cols = [c for c in df_xgb_v2.columns if c not in ("timestamp", target_col)]
df_xgb_v2.head()


### 2d. Retrain improved XGBoost
Target: MAPE approx 0.79% - the paper's headline result

In [ ]:
train_mask_xgb_v2 = df_xgb_v2["timestamp"].dt.year <= 2023
test_mask_xgb_v2 = df_xgb_v2["timestamp"].dt.year == 2024

X_train_xgb_v2 = df_xgb_v2.loc[train_mask_xgb_v2, xgb_engineered_cols]
X_test_xgb_v2 = df_xgb_v2.loc[test_mask_xgb_v2, xgb_engineered_cols]
y_train_xgb_v2 = df_xgb_v2.loc[train_mask_xgb_v2, target_col]
y_test_xgb_v2 = df_xgb_v2.loc[test_mask_xgb_v2, target_col]

with mlflow.start_run(run_name='xgb_shap_engineered') as run:
    xgb_v2_model = XGBRegressor(
        n_estimators=800, max_depth=7, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        objective="reg:squarederror", random_state=42, n_jobs=-1,
    )
    xgb_v2_model.fit(X_train_xgb_v2, y_train_xgb_v2)
    xgb_v2_preds = xgb_v2_model.predict(X_test_xgb_v2)

    xgb_v2_metrics = compute_all_metrics(y_test_xgb_v2.values, xgb_v2_preds)
    print(f"XGB + SHAP features -> MAPE {xgb_v2_metrics['mape']:.2f}% (baseline was {xgb_baseline_metrics['mape']:.2f}%)")

    mlflow.log_params({"model": "XGBRegressor", "feature_set": "shap_engineered", "n_features": len(xgb_engineered_cols)})
    mlflow.log_metrics(xgb_v2_metrics)
    mlflow.xgboost.log_model(xgb_v2_model, "model")

    xgb_v2_run_id = run.info.run_id


---
## Experiment 3 - LightGBM

### 3a. Baseline LightGBM
Target: MAPE approx 5.26%

In [ ]:
from lightgbm import LGBMRegressor

with mlflow.start_run(run_name='lgbm_baseline') as run:
    lgbm_baseline_model = LGBMRegressor(
        n_estimators=500, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    )
    lgbm_baseline_model.fit(X_train, y_train)
    lgbm_baseline_preds = lgbm_baseline_model.predict(X_test)

    lgbm_baseline_metrics = compute_all_metrics(y_test.values, lgbm_baseline_preds)
    print(f"LightGBM baseline -> MAPE {lgbm_baseline_metrics['mape']:.2f}%")

    mlflow.log_params({"model": "LGBMRegressor", "feature_set": "baseline", "n_features": len(feature_cols_baseline)})
    mlflow.log_metrics(lgbm_baseline_metrics)
    mlflow.lightgbm.log_model(lgbm_baseline_model, "model")

    lgbm_baseline_run_id = run.info.run_id


### 3b. SHAP diagnosis on baseline LightGBM

In [ ]:
explainer_lgbm = shap.TreeExplainer(lgbm_baseline_model)
shap_values_lgbm = explainer_lgbm(X_test)

shap.summary_plot(shap_values_lgbm, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_lgbm_baseline.png", dpi=150)
plt.close()

with mlflow.start_run(run_id=lgbm_baseline_run_id):
    mlflow.log_artifact("shap_lgbm_baseline.png")


### 3c. Engineer SHAP-guided features for LightGBM
**Note the formulas differ from XGBoost's version of the same-named features:**
- load_spike_vs_mean = (load - load_roll_mean_168) / (load_roll_std_168 + eps) (168h window, not 24h)
- temp_spike_vs_mean = tmax - tavg (unnormalized here)
- lag_24_x_hour = load_lag_24 x hour
- is_extreme_heat_event = 1[tmax > 95th percentile]

In [ ]:
from engineered_features import build_lgbm_engineered_features

df_lgbm_v2 = build_lgbm_engineered_features(df_feat)
lgbm_engineered_cols = [c for c in df_lgbm_v2.columns if c not in ("timestamp", target_col)]
df_lgbm_v2.head()


### 3d. Retrain improved LightGBM
Target: MAPE approx 0.9-1.0% (paper's text and table disagree slightly: 0.98% vs 0.91% - don't chase an exact match)

In [ ]:
train_mask_lgbm_v2 = df_lgbm_v2["timestamp"].dt.year <= 2023
test_mask_lgbm_v2 = df_lgbm_v2["timestamp"].dt.year == 2024

X_train_lgbm_v2 = df_lgbm_v2.loc[train_mask_lgbm_v2, lgbm_engineered_cols]
X_test_lgbm_v2 = df_lgbm_v2.loc[test_mask_lgbm_v2, lgbm_engineered_cols]
y_train_lgbm_v2 = df_lgbm_v2.loc[train_mask_lgbm_v2, target_col]
y_test_lgbm_v2 = df_lgbm_v2.loc[test_mask_lgbm_v2, target_col]

with mlflow.start_run(run_name='lgbm_shap_engineered') as run:
    lgbm_v2_model = LGBMRegressor(
        n_estimators=800, num_leaves=95, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    )
    lgbm_v2_model.fit(X_train_lgbm_v2, y_train_lgbm_v2)
    lgbm_v2_preds = lgbm_v2_model.predict(X_test_lgbm_v2)

    lgbm_v2_metrics = compute_all_metrics(y_test_lgbm_v2.values, lgbm_v2_preds)
    print(f"LightGBM + SHAP features -> MAPE {lgbm_v2_metrics['mape']:.2f}% (baseline was {lgbm_baseline_metrics['mape']:.2f}%)")

    mlflow.log_params({"model": "LGBMRegressor", "feature_set": "shap_engineered", "n_features": len(lgbm_engineered_cols)})
    mlflow.log_metrics(lgbm_v2_metrics)
    mlflow.lightgbm.log_model(lgbm_v2_model, "model")

    lgbm_v2_run_id = run.info.run_id


---
## Experiment 4 & 5 - LSTM / BiLSTM (manual benchmark, SHAP not applied)
Kept outside the SHAP loop and outside automated CI/CD promotion - logged to MLflow for comparison only, tagged clearly as a benchmark run.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Paper: "drew upon prior feature engineering insights from tree-based
# models—such as load_lag_24, load_roll_mean_24, and temp_spike_vs_mean"
lstm_feature_cols = [
    "load_lag_24", "load_roll_mean_24", "temp_spike_vs_mean",
    "tavg", "tmax", "hour", "day_of_week", "month",
]
WINDOW = 24  # hours of history per sequence — not published by the paper, our choice

lstm_df = df_xgb_v2[["timestamp", target_col] + lstm_feature_cols].dropna().reset_index(drop=True)

train_mask_seq = lstm_df["timestamp"].dt.year <= 2023
test_mask_seq = lstm_df["timestamp"].dt.year == 2024

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_seq_raw = x_scaler.fit_transform(lstm_df.loc[train_mask_seq, lstm_feature_cols])
X_test_seq_raw = x_scaler.transform(lstm_df.loc[test_mask_seq, lstm_feature_cols])
y_train_seq_raw = y_scaler.fit_transform(lstm_df.loc[train_mask_seq, [target_col]])
y_test_seq_raw = y_scaler.transform(lstm_df.loc[test_mask_seq, [target_col]])

def make_sequences(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i - window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_lstm, y_train_lstm = make_sequences(X_train_seq_raw, y_train_seq_raw)
X_test_lstm, y_test_lstm = make_sequences(X_test_seq_raw, y_test_seq_raw)

with mlflow.start_run(run_name='lstm_benchmark') as run:
    mlflow.set_tag('role', 'manual_benchmark')

    lstm_model = models.Sequential([
        layers.Input(shape=(WINDOW, len(lstm_feature_cols))),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    lstm_model.compile(optimizer="adam", loss="mse")

    early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    lstm_history = lstm_model.fit(
        X_train_lstm, y_train_lstm,
        validation_split=0.1, epochs=50, batch_size=256,
        callbacks=[early_stop], verbose=1,
    )

    lstm_preds_scaled = lstm_model.predict(X_test_lstm)
    lstm_preds = y_scaler.inverse_transform(lstm_preds_scaled).flatten()
    lstm_y_true = y_scaler.inverse_transform(y_test_lstm).flatten()

    lstm_metrics = compute_all_metrics(lstm_y_true, lstm_preds)
    print(f"LSTM -> MAPE {lstm_metrics['mape']:.2f}%")

    mlflow.log_params({"model": "LSTM", "window": WINDOW, "n_features": len(lstm_feature_cols)})
    mlflow.log_metrics(lstm_metrics)
    mlflow.tensorflow.log_model(lstm_model, "model")

    lstm_run_id = run.info.run_id


In [ ]:
with mlflow.start_run(run_name='bilstm_benchmark') as run:
    mlflow.set_tag('role', 'manual_benchmark')

    bilstm_model = models.Sequential([
        layers.Input(shape=(WINDOW, len(lstm_feature_cols))),
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Dropout(0.2),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    bilstm_model.compile(optimizer="adam", loss="mse")

    early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    bilstm_history = bilstm_model.fit(
        X_train_lstm, y_train_lstm,
        validation_split=0.1, epochs=50, batch_size=256,
        callbacks=[early_stop], verbose=1,
    )

    bilstm_preds_scaled = bilstm_model.predict(X_test_lstm)
    bilstm_preds = y_scaler.inverse_transform(bilstm_preds_scaled).flatten()

    bilstm_metrics = compute_all_metrics(lstm_y_true, bilstm_preds)
    print(f"BiLSTM -> MAPE {bilstm_metrics['mape']:.2f}% (LSTM was {lstm_metrics['mape']:.2f}%)")

    mlflow.log_params({"model": "BiLSTM", "window": WINDOW, "n_features": len(lstm_feature_cols)})
    mlflow.log_metrics(bilstm_metrics)
    mlflow.tensorflow.log_model(bilstm_model, "model")

    bilstm_run_id = run.info.run_id


---
## 6. Compare all runs
Pull metrics across all logged runs in this experiment for a side-by-side comparison table.

In [ ]:
runs_df = mlflow.search_runs(experiment_names=['electricity-distribution-forecast'])

comparison_cols = ["tags.mlflow.runName", "metrics.mape", "metrics.rmse", "metrics.mae", "metrics.peak_mape"]
comparison = (
    runs_df[[c for c in comparison_cols if c in runs_df.columns]]
    .dropna(subset=["metrics.mape"])
    .sort_values("metrics.peak_mape")
    .reset_index(drop=True)
)
comparison


## 7. Register best model + set champion alias
Once you've picked a winner (gated on Peak-MAPE per the project's promotion rule), register it and set the `champion` alias - this is metadata-only in Postgres, the R2 artifact doesn't move.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Gated on Peak-MAPE per the project's promotion rule — check `comparison`
# above and update this to whichever run_id actually wins before running.
best_run_id = xgb_v2_run_id
model_uri = f"runs:/{best_run_id}/model"

registered_model = mlflow.register_model(model_uri, "shef-day-ahead-model")
client.set_registered_model_alias("shef-day-ahead-model", "champion", registered_model.version)

print(f"Registered version {registered_model.version} of shef-day-ahead-model as 'champion'")
